# Benchmark Evaluation of the SSL Encoder Shareability Metric

In [81]:
import pandas as pd
from pathlib import Path
import numpy as np
import torch
from sklearn.decomposition import PCA
import copy


In [82]:
torch.manual_seed(42)
rng = np.random.default_rng(42)

## Shareability Metric

The SSL encoder shareability score is computed from the cross-covariance matrix

$$
M = \frac{1}{n} y^\top x.
$$

The shared encoder optimum uses only the symmetric part of M

$$
S = \frac{M + M^\top}{2}.
$$

The shared encoder optimum is 

$$
J_{shared}^* = max_i|\lambda_i(S)|.
$$

The seperate encoder optimum is 

$$
J_{sep}^* = \sigma_1(M).
$$

Giving the shareability metric

$$
p(M) = \frac{J_{shared}^*}{J_{sep}^*} = \frac{max_i|\lambda_i(S)|}{\sigma_1(M)}.
$$

In [83]:
def ssl_encoder_shareability(observation_1, observation_2):
        M = (observation_2.T @ observation_1) / observation_1.shape[0]
        S = (M + M.T) / 2
        
        S_eigenvalues = np.linalg.eigvalsh(S)
        J_shared = np.max(np.abs(S_eigenvalues))
        
        J_sep = np.linalg.svd(M, compute_uv=False)[0]
        
        P = J_shared / J_sep
        
        return P

## Whitening and Centering

Each observation is centered and whitened before computing the shareability metric. Centering removes the feature mean while whiteing rescales and decorrelates the observation so that scale and correlation do not dominate the shareability calculation.

For observation $x$,

$$
\tilde{x} = \Sigma_x^\frac{-1}{2}(x - \mu_x),
$$

where $\mu$ deontes the mean vector and $\Sigma$ denotes the covariance matrix estimated from the training data respectively.

In [84]:
class WhitenAndCenter:
        def __init__(self):
                self.mean = None
                self.inv_sqrt_cov = None

        def fit(self, x):
                self.mean = np.mean(x, axis=0)
                covariance = np.cov(x, rowvar=False)
                covariance_eigenvalues, covariance_eigenvectors = np.linalg.eigh(covariance)
                covariance_eigenvalues = np.clip(covariance_eigenvalues, 1e-8, None)
                self.inv_sqrt_cov = covariance_eigenvectors @ np.diag(1 / np.sqrt(covariance_eigenvalues)) @ covariance_eigenvectors.T
                return self

        def transform(self, x):
                return (x - self.mean) @ self.inv_sqrt_cov
        
        def fit_transform(self, x):
                return self.fit(x).transform(x)

## Shared Encoder

The linear shared encoder is given by

$$
z_x = w^\top x \qquad z_y = w^\top y.
$$

In [85]:
class SharedEncoder(torch.nn.Module):
    def __init__(self, vector_size):
        super().__init__()
        self.w = torch.nn.Linear(vector_size, 1, bias=False)
        
    def forward(self, x):
        return self.w(x)

## Seperate Encoder

The linear seperate encoder is given by 

$$
z_x = a^\top x \qquad z_y = b^\top y.
$$

In [86]:
class SeperateEncoder(torch.nn.Module):
    def __init__(self, vector_size):
        super().__init__()
        self.a = torch.nn.Linear(vector_size, 1, bias=False)
        self.b = torch.nn.Linear(vector_size, 1, bias=False)
        
    def forward(self, x, y):
        return self.a(x), self.b(y)

## Benchmark Dataset

The benchmark evaluation uses the **Multiple Features (MFeat)** dataset from the UCI machine learning repository. The dataset contains multiple feature representations of the same handwritten digit observations. This evaluation uses the Fourier (`mfeat-fou`) and Karhunen--Loeve (`mfeat-kar`) feature views because they provided mathematically distinct representations of the same handwritten digit observations.



In [87]:
fou = pd.read_csv(Path().cwd().resolve() / "mfeat-fou", sep=r"\s+", header=None)
kar = pd.read_csv(Path().cwd().resolve() / "mfeat-kar", sep=r"\s+", header=None)

fou = np.array(fou)
kar = np.array(kar)

print(len(fou))
print(len(kar))

2000
2000


## Dataset Split

The paired datasets are shuffled using a shared set of indices to preserve correspondence between the views, then divided 60/20/20 training, validation, holdout.

In [88]:
indices = np.arange(len(fou))
rng.shuffle(indices)

train_idx = int(len(fou) * 0.60)
val_idx = train_idx + int(len(fou) * 0.20)

train = indices[:train_idx]
val = indices[train_idx:val_idx]
test = indices[val_idx:]

fou_train = fou[train]
kar_train = kar[train]

fou_val = fou[val]
kar_val = kar[val]

fou_test = fou[test]
kar_test = kar[test]

print(f"fou train: {fou_train.shape}")
print(f"kar train: {kar_train.shape}")

print(f"fou val: {fou_val.shape}")
print(f"kar val: {kar_val.shape}")

print(f"fou test: {fou_test.shape}")
print(f"kar test: {kar_test.shape}")

fou train: (1200, 76)
kar train: (1200, 64)
fou val: (400, 76)
kar val: (400, 64)
fou test: (400, 76)
kar test: (400, 64)


## Dimensionality Reduction

The fourier view $x_{fou} \in \mathbb{R}^{76}$ and Karhunen--Loeve view $x_{kar} \in \mathbb{R}^{64}$ are independently reduced using PCA to a common dimensionality of $\mathbb{R}^{32}$.

In [89]:
pca_fou = PCA(n_components=32)
pca_kar = PCA(n_components=32)

fou_train = pca_fou.fit_transform(fou_train)
kar_train = pca_kar.fit_transform(kar_train)

fou_val = pca_fou.transform(fou_val)
kar_val = pca_kar.transform(kar_val)

fou_test = pca_fou.transform(fou_test)
kar_test = pca_kar.transform(kar_test)

print(f"fou train: {fou_train.shape}")
print(f"kar train: {kar_train.shape}")

print(f"fou val: {fou_val.shape}")
print(f"kar val: {kar_val.shape}")

print(f"fou test: {fou_test.shape}")
print(f"kar test: {kar_test.shape}")

fou train: (1200, 32)
kar train: (1200, 32)
fou val: (400, 32)
kar val: (400, 32)
fou test: (400, 32)
kar test: (400, 32)


In [90]:
wc_fou = WhitenAndCenter()
wc_kar = WhitenAndCenter()

fou_train = wc_fou.fit_transform(fou_train)
kar_train = wc_kar.fit_transform(kar_train)

fou_val = wc_fou.transform(fou_val)
kar_val = wc_kar.transform(kar_val)

fou_test = wc_fou.transform(fou_test)
kar_test = wc_kar.transform(kar_test)

## Shareability score

In [91]:
train_shareability = ssl_encoder_shareability(fou_train, kar_train)
val_shareability = ssl_encoder_shareability(fou_val, kar_val)
test_shareability = ssl_encoder_shareability(fou_test, kar_test)

print(f"Train Shareability: {train_shareability}")
print(f"Val Shareability: {val_shareability}")
print(f"Test Shareability: {test_shareability}")

Train Shareability: 0.81361930335706
Val Shareability: 0.756033312635282
Test Shareability: 0.753981612501371


In [92]:
fou_train = torch.tensor(fou_train, dtype=torch.float32)
kar_train = torch.tensor(kar_train, dtype=torch.float32)

fou_val = torch.tensor(fou_val, dtype=torch.float32)
kar_val = torch.tensor(kar_val, dtype=torch.float32)

fou_test = torch.tensor(fou_test, dtype=torch.float32)
kar_test = torch.tensor(kar_test, dtype=torch.float32)

## Shared encoder

As in the rank-one evaluation, the shared encoder is sensitive to initialization bias because the shared objective can contain multiple stationary directions. To reduce this sensitivity, the shared encoder is trained using 10 random restarts and the checkpoint with the lowest validation loss is retained.

In [93]:
best_restart_val = float("inf")
best_restart_state = None

for restart in range(10):
    shared_encoder = SharedEncoder(vector_size=32)
    shared_optimizer = torch.optim.SGD(shared_encoder.parameters(), lr=1e-1)

    epochs = 10000
    best_val_loss = float("inf")
    best_state = None
    for i in range(epochs):
        shared_encoder.train()
        Z_x = shared_encoder(fou_train)
        Z_y = shared_encoder(kar_train)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        train_loss = loss.item()
        shared_optimizer.zero_grad()
        loss.backward()
        shared_optimizer.step()
        with torch.no_grad():
            weights = shared_encoder.w.weight
            weights.div_(weights.norm(p=2))
                
        shared_encoder.eval()
        with torch.no_grad():
            Z_x = shared_encoder(fou_val)
            Z_y = shared_encoder(kar_val)
            val_loss = -torch.abs(torch.mean(Z_x * Z_y))
                
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(shared_encoder.state_dict())

    if best_val_loss < best_restart_val:
        best_restart_val = best_val_loss
        best_restart_state = best_state

print(f"Best Restart Loss: {best_restart_val}")

if best_restart_state is not None:
    shared_encoder.load_state_dict(best_restart_state)

Best Restart Loss: -0.7672738432884216


## Separate encoders

In [94]:
seperate_encoder = SeperateEncoder(vector_size=32)
seperate_optimizer = torch.optim.SGD(seperate_encoder.parameters(), lr=1e-1)

epochs = 10000
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    seperate_encoder.train()
    Z_x, Z_y = seperate_encoder(fou_train, kar_train)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    seperate_optimizer.zero_grad()
    loss.backward()
    seperate_optimizer.step()
    
    with torch.no_grad():
        current_weights = seperate_encoder.a.weight
        future_weights = seperate_encoder.b.weight
        
        current_weights.div_(current_weights.norm(p=2)) 
        future_weights.div_(future_weights.norm(p=2))
             
    seperate_encoder.eval()
    with torch.no_grad():
        Z_x, Z_y = seperate_encoder(fou_val, kar_val)
        val_loss = -torch.abs(torch.mean(Z_x * Z_y))
            
            
            
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(seperate_encoder.state_dict())

if best_state is not None:
    seperate_encoder.load_state_dict(best_state)
    
print(f"Best Separate Loss: {best_val_loss}")

Best Separate Loss: -0.9811269640922546


In [95]:
with torch.no_grad():
    Z_x = shared_encoder(fou_test)
    Z_y = shared_encoder(kar_test)
    shared_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"Shared Holdout Loss: {shared_loss}")

with torch.no_grad():
    Z_x, Z_y = seperate_encoder(fou_test, kar_test)
    sep_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"Separate Holdout Loss: {sep_loss}")
        

Shared Holdout Loss: -0.6898124814033508
Separate Holdout Loss: -0.896628737449646


## Benchmark Evaluation Results

Percent error between the empirical encoder ratio and theoretical shareability metric is calculated as

$$
\text{Percent Error} = \frac{|empirical - theoretical|}{theoretical} \times 100
$$

In [96]:
empirical_ratio = shared_loss / sep_loss

train_percent_error = (abs(empirical_ratio - train_shareability) / train_shareability) * 100
val_percent_error = (abs(empirical_ratio - val_shareability) / val_shareability) * 100
test_percent_error = (abs(empirical_ratio - test_shareability) / test_shareability) * 100

print(f"Case Study Results\n")
print(f"Holdout Encoder Ratio: {empirical_ratio:16.4f}\n")
print(f"{'Split':<12}{'Theoretical':>12}{'Percent Error':>15}")
print(f"{'Train':<12}{train_shareability:12.4f}{train_percent_error:14.2f}%")
print(f"{'Val':<12}{val_shareability:12.4f}{val_percent_error:14.2f}%")
print(f"{'Test':<12}{test_shareability:12.4f}{test_percent_error:14.2f}%")


Case Study Results

Holdout Encoder Ratio:           0.7693

Split        Theoretical  Percent Error
Train             0.8136          5.44%
Val               0.7560          1.76%
Test              0.7540          2.04%
